In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from functools import partial
import torch
torch.autograd.set_detect_anomaly(True)
from PIL import Image
import numpy as np
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from transformers import ProcessorMixin, MllamaProcessor, AutoTokenizer, AutoImageProcessor
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import TrainingArguments
from transformers import Trainer
from peft import LoraConfig, get_peft_model

from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder

from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from datasets import load_dataset, Dataset

from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.segmentation import SemanticSegmentationTool

from torchmetrics import JaccardIndex

from src.kitti_tracking import KittiDataset
from src.kitti_tracking_hf import KittiHFIterableDataset
from arc_trainer import ArcTrainer
from arc_utils import ARCCollator, ArcProcessor, ExtendedLMHead, ExtendEmbedding

/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/leonard/anaconda3/envs/agentic/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [3]:
root_dir = "/mnt/ssd/kitti_tracking"
n_steps, n_pred_steps = 1, 0

dataset_builder = KittiHFIterableDataset(
	root_dir=root_dir,
	split="training",
	n_steps=n_steps,
	n_pred_steps=n_pred_steps,
	transform=T.Compose([
		T.Resize((120, 320)),
	])
)
dataset = dataset_builder.to_hf_dataset()

In [4]:
vq_enc: Encoder = load_model(f"./checkpoints/encoder.pkl", "cpu")
vq_enc.eval()

clip_segm_name = "CIDAS/clipseg-rd64-refined"
clip_segm_processor = CLIPSegProcessor.from_pretrained(clip_segm_name, use_fast=True)
clip_segm_model = CLIPSegForImageSegmentation.from_pretrained(clip_segm_name)#.to(device)
clip_segm_model.eval()
clip_segm_size = (352, 352)

In [5]:
model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"

new_tokens = [
	f"<|vq_{i}|>" for i in range(vq_enc.blocks[-1].conv.w.shape[0])
] + ["<|begin_of_mask|>", "<|end_of_mask|>", "<|image|>"]

tokenizer = AutoTokenizer.from_pretrained("./checkpoints")
tokenizer.bom_token = "<|begin_of_mask|>"
tokenizer.eom_token = "<|end_of_mask|>"
tokenizer.add_tokens(new_tokens)

processor = MllamaProcessor(
    # processor.image_processor,
    AutoImageProcessor.from_pretrained(model_id),
    tokenizer
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [6]:
collator = ARCCollator(
    processor, processor.tokenizer,
    vq_enc
)

In [7]:
for i, sample in enumerate(dataset):
    break

In [9]:
collator([sample], prompts=['people', 'vehicles'])

z_logits torch.Size([1, 8192, 22, 22])
tensor([[128000, 128006,    882, 128007,    271,   -100,  30059,   4754,   2217,  13918,     13,    220, 128009,    271, 128006,  78191, 128007,    271, 136448, 131327, 128848, 130581, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 131327, 130581,
         134514, 131327, 134514, 130581, 132778, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 131496, 135707, 136196, 132926, 134514, 132778, 132778, 131476, 130030, 135707, 135707, 135707, 135707, 135707, 135707, 135707, 135707, 135707, 135707, 135707,
         135707, 135707, 135707, 135707, 131476, 136196, 130581, 132778, 132778, 130030, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 133385, 130030, 131496, 131516, 132778, 132778, 130030, 130030, 130030, 134640, 134640, 134640, 134640, 134640

IndexError: list index out of range

In [ ]:
for i, sample in enumerate(dataset):
	print(f"\r{i}", end="")
	_imgs = sample["rgb"]
	if len(_imgs) != 1:
		print()
		print(i, len(_imgs))
i

In [ ]:
for k, v in processed.items():
    if torch.is_tensor(v):
        print(k, v.requires_grad)

In [7]:
# add more tokens to the vocabulary
# load tokenizers
# model_id = "meta-llama/Llama-3.2-11B-Vision"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False
# model.resize_token_embeddings(len(tokenizer))
model.lm_head = ExtendedLMHead.from_llama("./checkpoints", model)
model.language_model.embed_tokens = ExtendEmbedding.from_llama("./checkpoints", model)
new_vocab_size = len(processor.tokenizer) - 1
model.config.get_text_config().vocab_size = new_vocab_size
model.vocab_size = new_vocab_size

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        # 'down_proj', 'gate_proj', 'up_proj',
        # "embed_tokens", "lm_head",
    ],
    use_dora=True, # optional DoRA 
    init_lora_weights="gaussian"
)
for p in model.language_model.embed_tokens.base_embedding.parameters(): p.requires_grad = False
for p in model.lm_head.base_head.parameters(): p.requires_grad = False
for p in model.model.vision_model.parameters(): p.requires_grad = False
for p in model.model.multi_modal_projector.parameters(): p.requires_grad = False

for p in model.language_model.embed_tokens.extra_embedding.parameters(): p.requires_grad = True
for p in model.lm_head.base_head.parameters(): p.requires_grad = True

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# llm_processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.35it/s]


trainable params: 12,410,880 || all params: 10,749,756,963 || trainable%: 0.1155


In [9]:
training_args = TrainingArguments(
	max_steps=3000,
	output_dir='./results',
	logging_dir='./logs',
	# gradient_checkpointing=True,
	per_device_train_batch_size=1,
	per_device_eval_batch_size=1,
	# num_train_epochs=1,
	# logging_steps=10,
	# save_total_limit=2,
	# disable_tqdm=False,       # enable progress bar
    logging_strategy="steps",
    logging_steps=10,          # show every step
    logging_first_step=True,  # show step 0/1
    bf16=True,
    # use_cpu=True,
    remove_unused_columns=False,
	learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_grad_norm=1,
    gradient_accumulation_steps=8,
)
arc_trainer = Trainer(
	# ckpt_path="",
	model=model,
	args=training_args,
	train_dataset=dataset,
    data_collator=partial(collator, prompts=['people', 'vehicles'], application="autonomous driving"),
)
arc_trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


ValueError: Cannot use apply_chat_template because this processor does not have a chat template.

In [ ]:
arc_trainer.save_model("./final_model")

In [ ]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, param.grad.norm())

In [ ]:
# VQ-VAE
def preprocess(img: Image.Image) -> torch.Tensor:
	img = torch.unsqueeze(T.ToTensor()(img), 0)
	return map_pixels(img)  # (1 - 2 * 0.1) * x + 0.1

def vq_encode(image, model, dev):
	x = preprocess(image).to(dev)
	z_logits = model(x.to(dev))
	z = torch.argmax(z_logits, axis=1)
	
	return z

def vq_decode(codes, model):
	z = F.one_hot(codes, num_classes=enc.vocab_size).permute(0, 3, 1, 2).float()

	x_stats = model(z).float()
	x_rec = unmap_pixels(torch.sigmoid(x_stats[:, :3]))
	x_rec = T.ToPILImage(mode='RGB')(x_rec[0])

	return x_rec

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
enc: Encoder = load_model("./checkpoints/encoder.pkl", device)
dec: Decoder = load_model("./checkpoints/decoder.pkl", device)

In [ ]:
sem_segm_tool = SemanticSegmentationTool(model_name="CIDAS/clipseg-rd64-refined")

In [ ]:
# load "codewords" from VAE
new_tokens = [
    f"<vq_{i}>" for i in range(enc.blocks[-1].conv.w.shape[0])
] + ["<begin_mask>", "<end_mask>"]

processor.tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(processor.tokenizer))

In [ ]:
# load dataset for fine-tuning
root_dir = "C:/Users/ngoak/data/kitti_tracking"
n_steps, n_pred_steps = 16, 3


train_ds = KittiDataset(
	root_dir, "training",
	n_steps, n_pred_steps,
	inp_transforms=T.Compose([
		T.Resize((240, 640)),
	])
)
for sample_dict in train_ds:
	rgb_images, depth_images = sample_dict['rgb'], sample_dict['depth']
	
	# rgb_code =
	# rgb_code_str =
	break

In [ ]:
iou_metric = JaccardIndex(task="binary")
tgt_class = ["person"]
prompt = "Goal"
for image in rgb_images:
	_prompt = "<|image|>"

	sem_mask = sem_segm_tool(image, tgt_class)
	z = vq_encode(sem_mask.convert("RGB"), enc, device)
	_z = z[0].cpu().numpy()
	mask_s = "<begin_mask>" + "".join([
		f"<vq_{_z[i, j]}>"
		for i in range(_z.shape[0])
		for j in range(_z.shape[1])
	]) + "<end_mask>"

	_sem_mask = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	patch = Image.new("RGB", (50, 50), (255, 255, 255))
	sem_mask.paste(patch, (100, 100))
	_sem_mask2 = torch.round(torch.tensor(np.array(sem_mask)) / 255)
	iou = 1 - iou_metric(_sem_mask2, _sem_mask)
	
	_prompt = "<|image|>" +\
	f"Thought: To ensure novel information is included in the transmission, I generate mask that reduces curiosity.\n" +\
	f"Action: {mask_s}\n" +\
	f"Observation: {iou:6f}\n" 

	print(iou)

	break

In [ ]:
gt = "<|image1|>" +\
    f"<|begin_of_text|> The curious mask that marks regions that might posibly contain {','.join(tgt_class)}"# +\
    # f"{mask_s}\n"

gt

In [ ]:
_prompt

In [ ]:
sem_mask

In [ ]:
sem_mask

In [ ]:
x = preprocess(rgb_images[0]).to(model.device)
z_logits = enc(x.to(device))
z = torch.argmax(z_logits, axis=1)
z.shape
display(T.ToPILImage(mode='RGB')(x[0]))

In [ ]:
_z = z[0].cpu().numpy()
mask_s = "<begin_mask>" + "".join([
    f"<vq_{_z[i, j]}>"
    for i in range(_z.shape[0])
    for j in range(_z.shape[1])
]) + "<end_mask>"


In [ ]:
processor.tokenizer(mask_s, return_tensors="pt", ).input_ids

In [ ]:
z = vq_encode(rgb_images[0], enc, device)

x_rec = vq_decode(z, dec)

display(x_rec)

In [ ]:
import requests


url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# prompt = "<|image|><|begin_of_text|>If I had to write a haiku for this one"
# inputs = processor(image, prompt, return_tensors="pt").to(model.device)

# output = model.generate(**inputs, max_new_tokens=30)

In [ ]:
def preprocess_fn(samples, clip_segm, vq_enc, ):
	
	def vq_preprocess(img: Image.Image) -> torch.Tensor:
		img = torch.unsqueeze(T.ToTensor()(img), 0)
		return map_pixels(img)  # (1 - 2 * 0.1) * x + 0.1
	
	images = sample["rgb"]   # list of PIL or arrays
	prompts = ["object"]     # or your actual prompts

	processed = processor(
		prompts=prompts,
		images=images,
		return_tensors="pt",
		padding=True,
		truncation=True
	)
	_pixel_values = processed.get("pixel_values", None)

	# labels for causal LM
	labels = processed["input_ids"].clone()

	# ignore padding in loss
	labels[labels == processor.tokenizer.pad_token_id] = -100

	return {
		"input_ids": processed["input_ids"][0].clone().detach(),
		"attention_mask": processed["attention_mask"][0].clone().detach(),
		"pixel_values": _pixel_values[0].clone().detach() if _pixel_values is not None else None,
		"labels": labels.clone().detach(),  # causal LM
		"aspect_ratio_ids": processed["aspect_ratio_ids"][0].clone().detach(),  # causal LM
		"aspect_ratio_mask": processed["aspect_ratio_mask"][0].clone().detach(),  # causal LM
	}